<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/05-data-sources-io/04-small-files-problem.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 5.4 — The small-files problem

We manufacture the disease locally (2,000 tiny files), measure it, then
apply the fixes. Local SSD softens the numbers — on S3/ADLS every gap
you see here is 10–100x wider.

In [ ]:
import os, sys, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("5.4-small-files")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

data = spark.range(2_000_000).select(
    col("id"),
    (col("id") % 100).alias("bucket"),
    F.round(F.rand(seed=1) * 100, 2).alias("value"),
)

## Manufacture the disease: same data, 2,000 files vs 4 files

In [ ]:
t0 = time.perf_counter()
data.repartition(2000).write.mode("overwrite").parquet("output/fragmented")
t_frag_write = time.perf_counter() - t0

t0 = time.perf_counter()
data.repartition(4).write.mode("overwrite").parquet("output/healthy")
t_healthy_write = time.perf_counter() - t0

import pathlib
def stats(root):
    files = list(pathlib.Path(root).rglob("*.parquet"))
    total = sum(f.stat().st_size for f in files)
    return len(files), total // 1024, (total // len(files)) // 1024

for name in ("fragmented", "healthy"):
    n, kb, avg = stats(f"output/{name}")
    print(f"{name:<11} {n:>5} files  {kb:>7} KB total  avg {avg:>5} KB/file")
print(f"\nwrite times: fragmented {t_frag_write:.1f}s vs healthy {t_healthy_write:.1f}s")

## The read-side tax

In [ ]:
def timed_read(path, label):
    t0 = time.perf_counter()
    n = spark.read.parquet(path).agg(F.sum("value")).collect()[0][0]
    print(f"{label:<11} full-scan agg: {time.perf_counter() - t0:5.2f}s  (sum={n:.0f})")

timed_read("output/fragmented", "fragmented")
timed_read("output/healthy",   "healthy")
# also check the two jobs' input stages in the UI: task counts and per-task input size

In [ ]:
# Bin-packing in action: 2000 files does NOT mean 2000 tasks —
# maxPartitionBytes + openCostInBytes pack small files together:
frag = spark.read.parquet("output/fragmented")
print("input partitions for 2000 files:", frag.rdd.getNumPartitions())
print("maxPartitionBytes:", spark.conf.get("spark.sql.files.maxPartitionBytes"))
print("openCostInBytes:  ", spark.conf.get("spark.sql.files.openCostInBytes"))

## Pushdown degrades on tiny files: stats-per-file cover only themselves

In [ ]:
# 'id BETWEEN x AND y' can skip most 500K-row row-groups in the healthy layout
# (ids are clustered by write partition); in the fragmented layout after
# repartition(2000) hashing, every tiny file spans the whole id range.
t0 = time.perf_counter()
spark.read.parquet("output/healthy").filter(col("id").between(100, 200)).count()
print(f"healthy    range-filter: {time.perf_counter() - t0:5.2f}s")

t0 = time.perf_counter()
spark.read.parquet("output/fragmented").filter(col("id").between(100, 200)).count()
print(f"fragmented range-filter: {time.perf_counter() - t0:5.2f}s")

## The cure: compaction

In [ ]:
t0 = time.perf_counter()
spark.read.parquet("output/fragmented").repartition(4) \
    .write.mode("overwrite").parquet("output/compacted")
print(f"compaction took {time.perf_counter() - t0:.1f}s")

n, kb, avg = stats("output/compacted")
print(f"compacted: {n} files, avg {avg} KB")
timed_read("output/compacted", "compacted")
# production version: swap directories atomically after, or use Delta OPTIMIZE (Module 15)

## Cleanup

In [ ]:
import shutil
shutil.rmtree("output", ignore_errors=True)
print("cleaned up.")

## Exercises

1. Compute the right file count for: 40 GB of output, 512 MB targets. Now express it in code for a DataFrame `df` about to write.
2. Simulate the incremental-append cause: write `data.limit(10_000)` with `mode("append")` into one directory 20 times in a loop. File count? Average size? Compact it.
3. Halve `spark.sql.files.maxPartitionBytes` and re-check `getNumPartitions()` on the fragmented read. Predict before running.
4. From lesson 5.2's explosion demo: how many files would `repartition(8)` + `partitionBy("bucket")` (100 values) produce from this data, worst case? Verify if you dare, then write the aligned fix.

In [ ]:
# your exercise code here
